## Reasoning Patch

In [1]:
%load_ext autoreload
%autoreload 2

### Overview

### Set-up

In [2]:
import torch
import gc
import random
import pandas as pd
import re
from tqdm import tqdm

import sys
sys.path.append("src")
import _util
from _intervention import forward_with_cache, prepare_batch_token_intervention, batch_intervene, get_attention_freeze_hooks

In [3]:
_util.print_GPU_availbility()

CUDA is available: True
Available devices:
  GPU 0: NVIDIA RTX A5500
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |      0 B   |
|       from small pool |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |      0 B   |      0 B   |      0 B

In [4]:
model_type = "R1" # GPT-OSS or R1

if model_type == "GPT-OSS":
    model, tokenizer = _util.load_OSS()
elif model_type == "R1":
    model, tokenizer = _util.load_R1()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [5]:
prompt_type = "pre_result" # empty or pre_result or pre_final_sum or h or h_pre_result or h_pre_final_sum
if prompt_type:
    prompt_type = "_" + prompt_type

# Load the divided prompts dataset
if 'h' in prompt_type:
    divided_prompts = pd.read_csv(f"data/{model_type}/h_divided_prompts{prompt_type[2:]}.csv")
else:
    divided_prompts = pd.read_csv(f"data/{model_type}/divided_prompts{prompt_type}.csv")
divided_prompts["base_number"] = divided_prompts["base_number"].astype('Int64')
divided_prompts["source_number"] = divided_prompts["source_number"].astype('Int64')
print(f"loaded {len(divided_prompts)} divided prompts")

loaded 2816 divided prompts


## Frozen Attention Patching

In [6]:
LAYER = 0

header = list(divided_prompts.columns) + ['generated_text']
filepath = _util.create_csv_file(f"experiments/activation_intervention/output/{model_type}", f"attention_freeze{prompt_type}.csv", header, overwrite=True)

batch_size = 24
df = divided_prompts
# df = divided_prompts[divided_prompts['intervention_id'] == 25]

for i in tqdm(range(0, len(df), batch_size)):
    torch.cuda.empty_cache()
    gc.collect()
    batch_rows = df.iloc[i:i+batch_size]
    
    # Prepare batch of intervention prompts
    base_before = batch_rows['base_before'].tolist()
    base_number = batch_rows['base_number'].tolist()
    base_after = batch_rows['base_after'].tolist()
    source_before = batch_rows['source_before'].tolist()
    source_number = batch_rows['source_number'].tolist()
    source_after = batch_rows['source_after'].tolist()

    tokens, hook = prepare_batch_token_intervention(model, tokenizer, LAYER, base_before, base_number, base_after, source_before, source_number)
    input_length = tokens["input_ids"].shape[1]

    for j in range(3):
        attention_freeze_hooks = get_attention_freeze_hooks(model, tokens)
        with torch.no_grad():
            output = batch_intervene(model, tokens["input_ids"], attention_freeze_hooks + [hook], attention_mask=tokens["attention_mask"])
        pred_toks = output.logits[:,-1,:].argmax(dim=-1)
        tokens["input_ids"] = torch.cat([tokens["input_ids"], pred_toks.unsqueeze(-1)], dim=1)
        del output
    
    for j, (_, row) in enumerate(batch_rows.iterrows()):
        generated_text = tokenizer.decode(tokens["input_ids"][j,input_length:]).replace(tokenizer.pad_token[-1], "")
        _util.write_to_csv(filepath, row.to_list() + [generated_text])

    del tokens, hook
    torch.cuda.empty_cache()
    gc.collect()


  0%|                                                                                         | 0/118 [00:00<?, ?it/s]

100%|███████████████████████████████████████████████████████████████████████████████| 118/118 [08:15<00:00,  4.20s/it]
